## 0. 準備環境（直接執行，不必逐行看懂）
先依序執行下面兩格，安裝所需套件。環境已可用時會自動跳過。

**Colab 請分開執行。** 第一格安裝 Conda 後可能自動重啟；等重新連線，再執行第二格安裝 PyGMT 等套件。勿在安裝期間重複按執行。

此教材使用 PyGMT 0.17 / GMT 6.5。新的 Colab 執行環境仍需安裝。


In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas, numpy, ipywidgets, ipyleaflet, pyproj
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")

In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas, numpy, ipywidgets, ipyleaflet, pyproj
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    subprocess.check_call([
        "mamba", "install", "-y", "-c", "conda-forge",
        "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "pandas", "numpy", "ipywidgets", "ipyleaflet", "pyproj"
    ])
else:
    print("跳過 Colab 安裝。")


# 03｜AI 探索與作業

從一個想觀察的小問題出發，找資料與論文圖片，和 AI 討論如何用圖與圖說表達，再試試 A–B 剖面。

請先另存副本，完成環境設置後，由上往下執行。

[課前介紹](https://github.com/jimmy60504/pygmt-map-lab/blob/main/intro.md) · [課程首頁](https://github.com/jimmy60504/pygmt-map-lab)

[1｜基本地圖與地震](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/01_maps_earthquakes.ipynb) · [2｜地形與 3D](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/02_terrain_3d.ipynb) · [3｜AI 探索與作業](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/03_ai_exploration.ipynb)


### 常用快捷鍵

| 操作 | Windows／Linux | Mac |
| --- | --- | --- |
| 執行目前儲存格並移到下一格 | Shift + Enter | Shift + Enter |
| 取消／切換註解 | Ctrl + / | ⌘ + / |

把游標放在程式行，或選取多行，再按切換註解的快捷鍵，即可移除或加上行首的 `#`。只選程式行，不要連中文說明一起取消註解。

修改後按 **Shift + Enter** 看結果，等執行完成再繼續下一步。


## 5. 開始用 AI：先想問題，再決定怎麼畫

不用提出很創新或複雜的研究。先找一份資料，想一個你想觀察的小問題，再和 AI 討論怎麼篩選、排序、比較與呈現。

1. **先寫一句想法**：我想看看什麼？例如「把台灣地震依深度上色，看看是否呈現與隱沒帶相符的空間排列」。這是待探索的想法，不是預先確定的結論。
2. **想想資料怎麼整理**：要選哪個區域、時間或深度？換個排序、分組或剖面方向，會不會看出原本沒注意到的特徵？
3. **和 AI 討論畫法**：地圖、剖面或其他圖，哪種更能回答問題？哪些部分用 PyGMT 完成？也可以搭配其他資料處理或互動工具，不必只用 PyGMT。
4. **看圖，再修正說法**：圖能不能讓人一眼抓到重點？若沒有看到預期特徵，也可以如實呈現；不要為了符合猜想而只挑支持它的資料。

最後把「你想表達的文字」和「圖」放在一起：先用一句話引導作圖，完成後再改成符合實際結果的簡短圖說，交代想看什麼、圖上怎麼呈現、實際看到什麼。簡單幾句就好，不用寫成研究報告，也不用證明假設成立。

可以這樣開始和 AI 討論：

> 我有＿＿資料，想看看＿＿。請先和我討論要怎麼篩選、排序或比較，以及哪種圖最容易看出重點，再寫程式。作品要用到 PyGMT，但可搭配其他工具。請幫我檢查圖例、尺度和資料限制；若結果不支持原本的想法，也要保留並說明。



### AI 畫完後，檢查這張圖說清楚了嗎？

程式能跑、圖看起來漂亮，不代表讀者就能正確理解。把圖和圖說放在一起，檢查下面幾件事：

| 檢查項目 | 要注意什麼 |
| --- | --- |
| **主題與圖說** | 一眼能看出你想表達什麼嗎？圖說交代資料與範圍、如何呈現、實際觀察到什麼；分清楚觀測、模型與推論，不把原本的猜想直接寫成結論。 |
| **座標與單位** | 軸的名稱、單位、時間基準／時區是否清楚？是線性還是對數尺度？地圖要能定位與辨向，可用經緯網、北箭頭或比例尺等資訊依需要補足，不必每張都塞齊所有元素。 |
| **圖例與色條** | 點、線、符號、大小、顏色各代表什麼？數值色條要有單位；超出色階與缺資料怎麼表示？規模不等於震度。 |
| **尺度與子圖對照** | 多張圖的顏色、大小與座標尺度能直接比較嗎？A–B 方向、子圖編號與框選範圍是否一致？若尺度不同，要明確交代。 |
| **篩選與不確定性** | 哪些時間、區域或規模被保留？是否做過平滑、插值或正規化？空白不一定代表沒有現象，可能只是缺資料；需要時呈現誤差或模型限制。 |
| **可讀性** | 縮到實際觀看大小後，字、線和圖例還看得清楚嗎？大圓、標籤、等高線或陰影有沒有遮住重點？ |

**剖面拉伸：標出垂直誇大倍率（VE）**

淺部剖面可以拉伸垂直方向，讓起伏更清楚，但要標示例如 `VE = 5×`。先把水平與垂直的實際距離換成同一單位，再比較它們在圖上的長度比例；相同實際距離若垂直畫得比水平長五倍，就是五倍誇大。不能只拿圖框的高／寬當成 VE，也不能直接用拉伸後的圖量斷層傾角。旋轉的 3D 透視圖則另有視角影響，不要把 `zsize` 直接當成 VE。

**局部放大：讓讀者知道這一小塊在哪裡**

局部地圖可在角落加一張較大範圍的位置示意圖（inset），用小框標出主圖涵蓋的範圍；也可以在大範圍主圖框出一區，再用另一張圖放大細節。框線、連接線或 `(a)、(b)` 要能清楚對照，並注意放大前後是否使用不同尺度。縮圖以定位為主，不必堆滿資料。

**顏色：不要只靠「看起來漂亮」來選**

- 重要類別不要只靠紅、綠區分，也可搭配符號、線型或文字。
- 有大小次序的量、正負異常與不同類別，適合的色票不一樣；顏色變化應幫助讀者理解資料。
- 用色覺模擬檢查；灰階可輔助檢查，但不能取代色覺測試。社群常用的色票也不一定是最友善的選擇。

可參考 [Seismica 的色票建議](https://seismica.library.mcgill.ca/policies) 與 [Nature 圖像規範](https://research-figure-guide.nature.com/figures/preparing-figures-our-specifications/)。各期刊要求不同，投稿時再確認目標期刊的規定。

可以請 AI 協助逐項檢查，但座標、單位、VE、資料來源和圖說，仍要自己對照資料確認。


### 先逛逛論文的圖，找找靈感

不用一開始就讀懂整篇 paper。先到 [Google Scholar](https://scholar.google.com/) 搜尋 `seismic`、`earthquake` 或 `seismicity`，也可以加上 `Taiwan`、`subduction`、`cross section`、`waveform` 等地區或圖像關鍵字。或先用 Google 圖片搜尋，看到有興趣的圖，再回到原論文看圖說。

也可以直接逛這些期刊，挑一篇題目有興趣的文章，先翻圖片：

| 期刊入口 | 主要範圍 | 逛圖時可以找什麼 |
| --- | --- | --- |
| [SRL — Seismological Research Letters](https://pubs.geoscienceworld.org/srl) | 地震學及相關觀測、方法與應用 | 地震事件、測站、波形與資料展示。 |
| [BSSA — Bulletin of the Seismological Society of America](https://pubs.geoscienceworld.org/bssa) | 地震學與相關研究 | 地震分布、震源、地動與分析結果。 |
| [GJI — Geophysical Journal International](https://academic.oup.com/gji) | 固體地球物理，不限地震 | 地下構造、剖面、波形與模型比較。 |
| [GRL — Geophysical Research Letters](https://agupubs.onlinelibrary.wiley.com/journal/19448007) | 地球與太空科學，不限地震 | 搜尋地震相關文章，看作者怎麼用少量圖呈現重點。 |
| [Seismica](https://seismica.library.mcgill.ca/) | 地震學與地震科學，開放取用 | 地震研究、資料與方法的各種呈現方式。 |

挑一張喜歡的圖就好，想想：**它想表達什麼？資料怎麼篩選或排列？我可以借用哪種畫法來表達自己的問題？** 重點是學呈現方式，不是照抄結論，也不必做出同樣複雜的研究。遇到付費文章，可找開放版本或換一篇。

把原論文連結與圖號留給自己，也可以給 AI 當討論參考。圖片搜尋只是入口，仍要回原文確認圖說；若要把原圖放進公開 GitHub，需確認授權並標明來源。



### 資料也可以換，找找新的靈感

地震資料不只有 USGS；先想清楚要的是「地震發生在哪裡」的目錄，還是「測站記錄到怎麼搖」的波形。

| 想找什麼 | 資料入口 | 可以做什麼 |
| --- | --- | --- |
| 台灣更細的地震資料 | [氣象署 GDMS](https://gdms.cwa.gov.tw/) | 查找台灣地震目錄與波形；想研究小地震或局部構造，可以從這裡找起，下載方式與權限依網站說明。 |
| 全球地震目錄 | [USGS](https://earthquake.usgs.gov/fdsnws/event/1/)／[ISC Bulletin](https://www.isc.ac.uk/iscbulletin/search/) | 取得時間、位置、深度與規模，畫分布圖或剖面；各目錄的收錄範圍與更新速度不同。 |
| 全球測站的地震波形 | [EarthScope（原 IRIS 服務）](https://service.earthscope.org/fdsnws/dataselect/1/) | 按測站、通道與時間下載波形，試做「震央與測站地圖＋波形」；不是每個測站都有所有時段的資料。 |

波形不是地震目錄，不能直接套進本課的地震點位程式。可以請 AI 協助讀取、處理，再用 PyGMT 呈現；例如畫出同一場地震在不同測站的記錄。


### 不只地震：把不同資料放在一起看

也可以加入地形、雨量、人口或土地覆蓋，探索一個簡單的小問題。

| 範圍 | 資料入口 | 可以想想的問題 |
| --- | --- | --- |
| 台灣 | [20 公尺數值地形模型](https://data.gov.tw/dataset/35430) | 換成較細的地形，能看出哪些山谷、盆地或地形邊界？ |
| 台灣 | [氣象署開放資料](https://opendata.cwa.gov.tw/index)：雨量與氣象觀測 | 同一場降雨，山區和平地的分布有何不同？ |
| 全球 | [GEBCO 海陸地形](https://www.gebco.net/data-products/gridded-bathymetry-data) | 海溝的位置與不同深度的地震如何對應？ |
| 全球 | [Global CMT 震源機制目錄](https://www.globalcmt.org/CMTfiles.html) | 不同區域的地震，斷層運動型態是否不同？ |
| 全球 | [WorldPop 人口網格](https://www.worldpop.org/) | 地震周邊的人口集中在哪裡？空間分布不等於災害風險。 |
| 全球 | [Copernicus 土地覆蓋](https://land.copernicus.eu/en/products/global-dynamic-land-cover/land-cover-2020-raster-10-m-global-annual) | 山地與平原的森林、農地、建成區分布有何差異？ |
| 全球 | [Natural Earth 基礎圖資](https://www.naturalearthdata.com/downloads/) | 加上國界、城市與河流，能否讓研究區域更容易理解？適合區域或全球圖，不是精細街道圖。 |

可以從「地震＋海底地形」、「地震＋震源機制」或「地形＋雨量」選一個方向。先選小區域、少量資料，把一個問題講清楚即可；不必把所有資料都放上圖。

下載前請 AI 一起檢查年份、座標系統、解析度、單位與授權；部分服務可能需要註冊或 API 金鑰。公開資料不代表格式能直接混用，資料疊在一起也不代表已證明因果關係。作品仍需用到 PyGMT，但可以搭配其他工具處理與呈現。


### AI 展示：在地圖點 A、B，切開地震分布
完成環境安裝後，本格可獨立執行，會自行下載 2020 年至今的 USGS 地震。

1. 在互動地圖點一下設定 **A**，再點一下設定 **B**。
2. 用「半寬 km」拉桿決定剖面線兩側要選多寬；藍色範圍就是取樣走廊。
3. 按「更新剖面」，先看標有 A、B、剖面線與取樣走廊的 PyGMT 地圖，再看由 A 往 B 的距離—深度圖。換位置前按「重選 A、B」。

可以先試 A 約在花蓮南方、B 在台灣東北方；不必限制南北向。地圖用 ipyleaflet 操作，PyGMT 負責地圖與剖面出圖，`project()` 計算沿線距離與離線距離（球面近似）。

**讀圖注意**：深度向下增加；超過 150 km 的事件沿用最深端藍色。水平與垂直比例不一定相同，不能直接量圖上的傾角。已下載的資料只涵蓋 119–123°E、21–26°N；在框外畫剖面不會自動取得新地震。地震排列也不是完整板塊邊界。

互動地圖需網路與運作中的 Colab／Jupyter；GitHub 靜態預覽無法點選。若 Colab 提示允許自訂元件，請確認後啟用。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [ipyleaflet Map](https://ipyleaflet.readthedocs.io/en/latest/map_and_basemaps/map.html) | 互動地圖與點擊事件 | `on_interaction`、`coordinates` |
| [pygmt.project()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.project.html) | 投影及篩選剖面地震 | `center`、`endpoint`、`width`、`length`、`unit` |
| [Jupyter Widgets](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html) | 拉桿與按鈕 | `IntSlider`、`Button`、`Output` |
| [Figure.coast()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.coast.html) | 地圖底圖 | `region`、`projection`、`land` |
| [Figure.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 地震、剖面線與走廊 | `x`、`y`、`close`、`pen` |
| [Figure.text()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.text.html) | A、B 標籤 | `text`、`offset`、`font` |


In [ ]:
import pygmt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipyleaflet import Map, Marker, Polyline, Polygon, CircleMarker, LayerGroup
from IPython.display import display, clear_output
from datetime import datetime, timezone
from pyproj import Geod

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass  # 本機 Jupyter 不需要 Colab 的元件管理器


# 1. 本格自行下載資料，不依賴其他 cell
endtime = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
    "&starttime=2020-01-01"
    f"&endtime={endtime}"
    "&minmagnitude=2"
    "&minlongitude=119&maxlongitude=123"
    "&minlatitude=21&maxlatitude=26"
)
quakes = pd.read_csv(url).dropna(subset=["longitude", "latitude", "depth", "mag"])

def magnitude_size(magnitude):
    if magnitude < 3:
        return 0.035
    elif magnitude < 4:
        return 0.07
    elif magnitude < 5:
        return 0.14
    elif magnitude < 6:
        return 0.28
    elif magnitude < 7:
        return 0.56
    else:
        return 1.12


# 2. 地圖只負責點選；GMT project 計算沿線距離與垂直距離
# 使用球面近似，走廊邊界也用同一種球面幾何
sphere = Geod(a=6371008.8, f=0)
points = []  # 點擊順序：A、B；每個座標為（緯度、經度）
map_view = Map(center=(24, 121.5), zoom=7, scroll_wheel_zoom=True,
               layout=widgets.Layout(height="450px"))
earthquake_layer = LayerGroup(layers=tuple(
    CircleMarker(location=(row.latitude, row.longitude), radius=2,
                 color="#777777", fill_color="#777777", fill_opacity=0.4, weight=0)
    for row in quakes.itertuples()
))
selection_layer = LayerGroup()
map_view.add(earthquake_layer)
map_view.add(selection_layer)

width_slider = widgets.IntSlider(value=40, min=10, max=100, step=5,
    description="半寬 km", continuous_update=False)
update_button = widgets.Button(description="更新剖面", button_style="primary")
reset_button = widgets.Button(description="重選 A、B")
status = widgets.HTML("請在地圖點 A，再點 B。灰點為已下載的地震。")
output = widgets.Output()


def profile_geometry():
    (lat_a, lon_a), (lat_b, lon_b) = points
    azimuth, _, meters = sphere.inv(lon_a, lat_a, lon_b, lat_b)
    if meters < 1000:
        raise ValueError("A、B 請至少相距 1 km。")
    along = np.linspace(0, meters, 60)
    lons, lats, back = sphere.fwd(
        np.full(60, lon_a), np.full(60, lat_a), np.full(60, azimuth), along
    )
    # 軌跡每個位置的前進方向，左右各延伸半寬
    left_lon, left_lat, _ = sphere.fwd(lons, lats, back + 90, np.full(60, width_slider.value * 1000))
    right_lon, right_lat, _ = sphere.fwd(lons, lats, back - 90, np.full(60, width_slider.value * 1000))
    track = list(zip(lats, lons))
    corridor = list(zip(left_lat, left_lon)) + list(zip(right_lat, right_lon))[::-1]
    return meters / 1000, track, corridor


# 3. 點選或改寬度時，更新地圖範圍並清掉舊剖面
def refresh_selection():
    with output:
        clear_output(wait=False)
    layers = [Marker(location=p, title=label, draggable=False) for p, label in zip(points, ["A", "B"])]
    if len(points) == 2:
        try:
            length, track, corridor = profile_geometry()
        except ValueError as error:
            status.value = str(error)
        else:
            layers += [Polygon(locations=corridor, color="#1565c0", fill_opacity=0.12),
                       Polyline(locations=track, color="#1565c0", weight=3)]
            status.value = f"A={points[0]}；B={points[1]}；長約 {length:.0f} km，全寬 {2*width_slider.value} km。請按更新剖面。"
    else:
        status.value = "請點 B。" if points else "請點 A，再點 B。"
    selection_layer.layers = tuple(layers)

def on_map_click(**event):
    if event.get("type") != "click":
        return
    if len(points) == 2:
        status.value = "要換位置，請先按「重選 A、B」。"
        return
    points.append(tuple(event["coordinates"]))
    refresh_selection()

def reset_selection(_):
    points.clear()
    refresh_selection()


# 4. 按鈕才觸發計算與 GMT 出圖；深度向下增加
def draw_section(_=None):
    with output:
        clear_output(wait=True)
        if len(points) != 2:
            print("請先在地圖選好 A、B。")
            return
        try:
            length, track, corridor = profile_geometry()
        except ValueError as error:
            print(error)
            return

        selected = pygmt.project(
            data=quakes[["longitude", "latitude", "depth", "mag"]],
            center=list(points[0][::-1]),  # GMT 順序：經度、緯度
            endpoint=list(points[1][::-1]),
            unit=True,  # 距離單位 km
            length="w",  # 只保留 A 到 B 之間
            width=[-width_slider.value, width_slider.value],  # 線兩側的半寬
            convention="xypqz",  # 經度、緯度、沿線距離、離線距離、深度、規模
        )
        if selected.empty:
            print("範圍內沒有地震，請改位置或加大半寬。")
            return
        selected.columns = ["longitude", "latitude", "distance", "offset", "depth", "mag"]
        selected = selected.sort_values("mag")
        print(f"取樣 {len(selected)} / {len(quakes)} 筆；2020-01-01 至 {endtime} UTC")
        print("只含下載範圍：119–123°E、21–26°N；走廊超出此範圍的部分沒有資料。")

        # 新增：先畫地理位置圖，灰點為全部地震，彩色點為剖面取樣
        map_fig = pygmt.Figure()
        pygmt.makecpt(cmap="gmt/seis", series=[0, 150, 1], continuous=True, background=True)
        corridor_lat, corridor_lon = np.array(corridor).T
        track_lat, track_lon = np.array(track).T
        map_fig.coast(
            region=[min(119, corridor_lon.min()-0.1), max(123, corridor_lon.max()+0.1),
                    min(21, corridor_lat.min()-0.1), max(26, corridor_lat.max()+0.1)],
            projection="M14c",
            land="gray95", water="aliceblue", shorelines="0.5p,gray40",
            frame=["af", "+tA-B profile location"],
        )
        map_fig.plot(x=quakes.longitude, y=quakes.latitude, style="c0.035c", fill="gray70")
        map_fig.plot(
            x=selected.longitude, y=selected.latitude,
            style="c", size=selected.mag.apply(magnitude_size),
            fill=selected.depth, cmap=True, transparency=40, pen="0.2p,gray30",
        )

        # 走廊只畫外框，避免蓋住地震的深度顏色
        map_fig.plot(x=corridor_lon, y=corridor_lat, close=True, pen="1p,blue,--")
        map_fig.plot(x=track_lon, y=track_lat, pen="1.5p,black")
        map_fig.plot(x=[points[0][1], points[1][1]], y=[points[0][0], points[1][0]],
                     style="s0.22c", fill="white", pen="1p,black")
        map_fig.text(
            x=[points[0][1], points[1][1]], y=[points[0][0], points[1][0]],
            text=["A", "B"], font="14p,Helvetica-Bold,black",
            offset="0.2c/0.2c", justify="BL", fill="white",
        )
        map_fig.colorbar(frame=["xaf", "y+lDepth (km)"])
        map_fig.show()

        # 再畫剖面：沿用同一套規模大小與 0–150 km 深度色階
        fig = pygmt.Figure()
        pygmt.makecpt(cmap="gmt/seis", series=[0, 150, 1], continuous=True, background=True)
        fig.basemap(
            region=[0, length, min(0, np.floor(selected.depth.min()/50)*50), max(50, np.ceil(selected.depth.max()/50)*50)],
            projection="X14c/-9c",
            frame=["xaf+lDistance from A (km)", "yaf+lDepth (km)", "+tA to B"],
        )
        fig.plot(
            x=selected.distance,
            y=selected.depth,
            style="c",
            size=selected.mag.apply(magnitude_size),
            fill=selected.depth,
            cmap=True,
            transparency=40,
            pen="0.2p,gray30",
        )
        fig.colorbar(frame=["xaf", "y+lDepth (km)"])
        fig.show()


map_view.on_interaction(on_map_click)
width_slider.observe(lambda change: refresh_selection(), names="value")
reset_button.on_click(reset_selection)
update_button.on_click(draw_section)
display(widgets.VBox([map_view, widgets.HBox([width_slider, update_button, reset_button]), status, output]))


## 6. 隔週繳交
這次作業就用 AI 做！想想你想呈現什麼，讓 AI 幫你把點子做出來。還沒靈感的話，可以先逛逛 [PyGMT Gallery](https://www.pygmt.org/v0.17.0/gallery/index.html)，找喜歡的範例，再試著改造或組合。

作品請把**圖＋一小段圖說**放在一起：說明你想看什麼、如何呈現，以及實際觀察到什麼。不必創新或複雜，也不必得到符合原先猜想的結果；重點是讓人一眼抓到你想表達的事。可以搭配其他工具，不必只用 PyGMT。

作業只需滿足兩個條件：

1. **作品與 PyGMT 有關。**
2. **將作品上傳 GitHub，繳交 repository 連結**，並確認教師能開啟。

題材、區域、呈現形式與圖的張數都可自由發揮，不限定沿用課堂範例；隔週繳交即可。


## 資料來源與版本

- [原始課程參考 Notebook](https://github.com/oceanicdayi/plot_plate_boundary_pygmt/blob/main/pygmt_plot_plate_boundary.ipynb)
- [GMT 全球地形資料](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)：PyGMT 載入，首次使用需連網。
- [PyGMT 0.17 安裝文件](https://www.pygmt.org/v0.17.0/install.html)


| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 本課使用的真實事件資料 | 查詢條件包含在下載網址中 |

